In [ ]:

!pip install langchain-gigachat
!pip install langchain_community
!pip install langchain_google_genai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 69.9/69.9 kB 2.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 458.9/458.9 kB 17.0 MB/s eta 0:00:00
  Attempting uninstall: langchain-core
    Found existing installation: langchain-core 1.2.6
    Uninstalling langchain-core-1.2.6:
      Successfully uninstalled langchain-core-1.2.6
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langgraph-prebuilt 1.0.5 requires langchain-core>=1.0.0, but you have langchain-core 0.3.83 which is incompatible.
langchain 1.2.3 requires langchain-core<2.0.0,>=1.2.1, but you have langchain-core 0.3.83 which is incompatible.
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.5/2.5 MB 40.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.0/1.0 MB 59.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 490.2/490.2 kB 30.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━

In [ ]:
!pip install groq
!pip install langchain_groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 7.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 137.5/137.5 kB 7.4 MB/s eta 0:00:00
  Attempting uninstall: groq
    Found existing installation: groq 1.0.0
    Uninstalling groq-1.0.0:
      Successfully uninstalled groq-1.0.0


In [ ]:
from langchain_gigachat.chat_models import GigaChat
from langchain_core.messages import HumanMessage,SystemMessage
import  os
from langchain_community.embeddings import GigaChatEmbeddings
import  google.generativeai  as  genai
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_core.prompts import PromptTemplate

import numpy as np
import pandas as pd
import json
import copy
from typing import List, Tuple, Dict, Any, Set
import random
from itertools import combinations

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


In [ ]:
class PeerToPeerConsensusAHP:
    def __init__(self, threshold: float = 1.03, max_iterations: int = 10):
        self.threshold = threshold
        self.max_iterations = max_iterations
        self.current_pcms = None
        self.dm_ids = None
        self.alternatives = None

    def compatibility_index(self, A: np.ndarray, B: np.ndarray) -> float:
        """
        Индекс совместимости между двумя матрицами
        """
        n = A.shape[0]
        hadamard_product = A*B.T
        return (1 / n**2) * np.sum(hadamard_product)

    def calculate_ici_matrix(self, all_pcms: List[np.ndarray], D: Set[Tuple[int, int]]) -> np.ndarray:
        """
        Матрица консенсуса - формула 6
        """
        m = len(all_pcms)
        ici_matrix = np.ones((m, m))

        for i in range(m):
            for j in range(m):
                if i == j:
                    ici_matrix[i, j] = 1.0
                elif (i, j) in D:
                    ici = self.compatibility_index(all_pcms[i], all_pcms[j])
                    ici_matrix[i, j] = ici
                else:
                    ici_matrix[i, j] = np.nan

        return ici_matrix

    def find_max_ici_pair(self, ici_matrix: np.ndarray, D: Set[Tuple[int, int]]) -> Tuple[int, int]:
        """
        Нахождение пары с максимальным ICI из доступных пар
        """
        max_ici = -1
        best_pair = None

        for pair in D:
            i, j = pair
            if not np.isnan(ici_matrix[i, j]) and ici_matrix[i, j] > max_ici:
                max_ici = ici_matrix[i, j]
                best_pair = pair
        if best_pair is not None:
            best_pair = tuple(sorted(best_pair))
        return best_pair

    def calculate_alpha(self, ici_matrix: np.ndarray, pair: Tuple[int, int], m: int) -> Tuple[float, float]:
        """
        Расчет коэффициентов по формулам 11-12
        """
        a, b = pair

        sum_ici_a = 0
        sum_ici_b = 0
        count = 0

        for l in range(m):
            if l != a and l != b and not np.isnan(ici_matrix[a, l]) and not np.isnan(ici_matrix[b, l]):
                sum_ici_a += ici_matrix[a, l]
                sum_ici_b += ici_matrix[b, l]
                count += 1

        if count == 0 or (sum_ici_a + sum_ici_b) == 0:
            return 0.5, 0.5

        total = sum_ici_a + sum_ici_b
        alpha_a = 1 - (sum_ici_a / (2 * total))
        alpha_b = 1 - (sum_ici_b / (2 * total))

        return round(alpha_a,4), round(alpha_b,4)

    def update_pcm(self, A_own: np.ndarray, A_other: np.ndarray, alpha: float) -> np.ndarray:
        """
        Обновление матрицы по формулам 9-10
        """
        return np.round(np.power(A_own, alpha) * np.power(A_other, 1 - alpha),3)

    def calculate_decision_maker_weights(self, ici_matrix: np.ndarray, m: int) -> np.ndarray:
        """
        Расчет весов экспертов через марковскую цепь - 7-8
        """
        P = np.zeros((m, m))

        for i in range(m):
            denominators = []
            for j in range(m):
                if i != j and not np.isnan(ici_matrix[i, j]):
                    denominators.append(1 / ici_matrix[i, j])

            if denominators:
                denominator_sum = sum(denominators)
                for j in range(m):
                    if i != j and not np.isnan(ici_matrix[i, j]):
                        P[i, j] = (1 / ici_matrix[i, j]) / denominator_sum
        n = P.shape[0]
        A = P.T - np.eye(n)
        A = np.vstack([A, np.ones(n)])
        b = np.zeros(n + 1)
        b[-1] = 1

        pi, _, _, _ = np.linalg.lstsq(A, b, rcond=None)
        weights = np.abs(pi) / np.abs(pi).sum()

        return weights

    def aggregate_group_pcm(self, all_pcms: List[np.ndarray], weights: np.ndarray) -> np.ndarray:
        n = all_pcms[0].shape[0]
        log_G = np.zeros((n, n))

        for k, A_k in enumerate(all_pcms):
            log_G += weights[k] * np.log(A_k)

        return np.exp(log_G)

    def calculate_priority_vector(self, A: np.ndarray) -> np.ndarray:
        n = A.shape[0]
        geometric_means = np.prod(A, axis=1) ** (1/n)
        return np.round(geometric_means / geometric_means.sum(),3)

    def check_consistency(self, A: np.ndarray) -> Tuple[float, float]:
        n = A.shape[0]

        RI_table = {1: 0, 2: 0, 3: 0.52, 4: 0.89, 5: 1.11,
                   6: 1.25, 7: 1.35, 8: 1.40, 9: 1.45, 10: 1.49}
        RI = RI_table.get(n, 1.49)

        eigenvalues, _ = np.linalg.eig(A)
        lambda_max = max(eigenvalues.real)

        CI = (lambda_max - n) / (n - 1) if n > 1 else 0
        CR = CI / RI if RI > 0 else 0

        return CI, CR

In [ ]:
def fix_pcm_by_eigenvector(self, A, max_changes=3):
    import numpy as np

    SAATY = np.array([1,2,3,4,5,6,7,8,9])

    def nearest_saaty(x):
        return SAATY[np.argmin(np.abs(SAATY - x))]

    n = A.shape[0]
    vals, vecs = np.linalg.eig(A)
    w = np.abs(vecs[:, np.argmax(vals.real)].real)
    w = w / w.sum()

    A_star = np.zeros_like(A, dtype=float)
    for i in range(n):
        for j in range(n):
            A_star[i, j] = w[i] / w[j]

    diffs = []
    for i in range(n):
        for j in range(i+1, n):
            diff = abs(np.log(A[i, j]) - np.log(A_star[i, j]))
            diffs.append((diff, i, j))
    diffs.sort(reverse=True)

    A_new = A.copy()
    changes = 0
    for _, i, j in diffs:
        if changes >= max_changes:
            break
        x_round = nearest_saaty(A_star[i, j])
        if x_round != A[i, j]:
            A_new[i, j] = x_round
            A_new[j, i] = 1.0 / x_round
            changes += 1
    return A_new

PeerToPeerConsensusAHP.fix_pcm_by_eigenvector = fix_pcm_by_eigenvector

In [ ]:
def enhanced_interactive_acceptance_decision(self, pair: Tuple[int, int], iteration: int,
                                           dm_ids: List[str], ici_matrix: np.ndarray, D: Set[Tuple[int, int]] ) -> Tuple[bool, bool]:
    a, b = pair
    ici_value = ici_matrix[a, b]

    print(f"\n{'='*80}")
    print(f"ИТЕРАЦИЯ {iteration + 1}")
    print(f"{'='*80}")

    # Показ матрицы ICI
    print(f"\nМАТРИЦА ИНДИВИДУАЛЬНЫХ ИНДЕКСОВ КОНСЕНСУСА (ICI):")
    self.print_ici_matrix(ici_matrix, dm_ids)

    alpha_a, alpha_b = self.calculate_alpha(ici_matrix, (a, b), len(dm_ids))

    proposed_A = self.update_pcm(self.current_pcms[a], self.current_pcms[b], alpha_a)
    proposed_B = self.update_pcm(self.current_pcms[b], self.current_pcms[a], alpha_b)

    print(f"\nПРЕДЛОЖЕНИЕ ПО ПЕРЕСМОТРУ:")
    print(f"   {dm_ids[a]} сохранит {alpha_a} своего мнения")
    print(f"   {dm_ids[b]} сохранит {alpha_b} своего мнения")

    can_a_change = (a, b) in D  # DM_a может изменить?
    can_b_change = (b, a) in D  # DM_b может изменить?

    # ---- ОПРОС ЭКСПЕРТА A (LLM вместо input) ----
    if can_a_change:
        print(f"\nПРЕДЛАГАЕМАЯ МАТРИЦА ДЛЯ {dm_ids[a]}:")
        self.print_matrix_comparison(
            self.current_pcms[a],
            proposed_A,
            f"Текущая {dm_ids[a]}",
            f"Предлагаемая {dm_ids[a]}",
        )

        # Привязка DM -> модель LLM (здесь предполагаем совпадение индексов)
        model_name = LLMMODELS[a % len(LLMMODELS)]

        accept_a, reason_a = llm_accept_change(model_name, dm_id=dm_ids[a],old_pcm=self.current_pcms[a], new_pcm=proposed_A,
    alternatives=self.alternatives,alpha=alpha_a)
        print(f" {dm_ids[a]} (модель {model_name}) решение: {'принял' if accept_a else 'отклонил'}")
        print(reason_a)
    else:
        accept_a = False  # Не спрашиваем

    # ---- ОПРОС ЭКСПЕРТА B (LLM вместо input) ----
    if can_b_change:
        print(f"\nПРЕДЛАГАЕМАЯ МАТРИЦА ДЛЯ {dm_ids[b]}:")
        self.print_matrix_comparison(
            self.current_pcms[b],
            proposed_B,
            f"Текущая {dm_ids[b]}",
            f"Предлагаемая {dm_ids[b]}",
        )

        model_name = LLMMODELS[b % len(LLMMODELS)]


        accept_b,reason_b = llm_accept_change(model_name=model_name,dm_id=dm_ids[b],old_pcm=self.current_pcms[b],
            new_pcm=proposed_B,alternatives=self.alternatives,alpha=alpha_b)
        print(f" {dm_ids[b]} (модель {model_name}) решение: {'принял' if accept_b else 'отклонил'}")
        print(reason_b)
    else:
        accept_b = False  # Не спрашиваем

    return accept_a, accept_b

def print_ici_matrix(self, ici_matrix: np.ndarray, dm_ids: List[str]):
    m = len(dm_ids)

    print("      " + "".join([f"{dm_id:>10}" for dm_id in dm_ids]))
    print("      " + "-" * (10 * m))

    for i in range(m):
        row_str = f"{dm_ids[i]:>5} |"
        for j in range(m):
            if i == j:
                row_str += f"{'1.0000':>10}"
            elif np.isnan(ici_matrix[i, j]):
                row_str += f"{'---':>10}"
            else:
                row_str += f"{ici_matrix[i, j]:>10.4f}"
        print(row_str)

    max_ici = np.nanmax(ici_matrix)
    print(f"   Максимальный: {max_ici:.4f} (наиболее несовместимая пара)")

def print_matrix_comparison(self, current: np.ndarray, proposed: np.ndarray, current_label: str, proposed_label: str):
    n = current.shape[0]
    col_width = 10

    # Заголовки
    print(f"   {current_label:<{col_width*n}} | {proposed_label}")
    print("   " + "-" * (col_width*n) + "-+-" + "-" * (col_width*n))

    # Вывод матриц построчно
    for i in range(n):
        current_row = ""
        for j in range(n):
            current_row += f"{current[i,j]:{col_width}.3f}"
        proposed_row = ""
        for j in range(n):
            proposed_row += f"{proposed[i,j]:{col_width}.3f}"

        print(f"   {current_row} | {proposed_row}")
        print()

def print_simple_matrix(self, matrix: np.ndarray):
    n = matrix.shape[0]
    for i in range(n):
        row = "   "
        for j in range(n):
            row += f"{matrix[i,j]:8.3f}"
        print(row)


# Добавляем методы к классу
PeerToPeerConsensusAHP.enhanced_interactive_acceptance_decision = enhanced_interactive_acceptance_decision
PeerToPeerConsensusAHP.print_ici_matrix = print_ici_matrix
PeerToPeerConsensusAHP.print_matrix_comparison = print_matrix_comparison
PeerToPeerConsensusAHP.print_simple_matrix = print_simple_matrix

In [ ]:
def clean_llm_response(raw: str) -> str:
    """Очищает ответ LLM от markdown и лишнего текста"""
    cleaned = raw.strip()

    # Markdown cleanup для ```
    if '```json' in cleaned:
        cleaned = cleaned.split('``````')[0].strip()
    elif '```' in cleaned:
        parts = cleaned.split('```')
        for part in parts:
            if '{' in part and '}' in part:
                cleaned = part.strip()
                break

    # Extract JSON block
    start = cleaned.find('{')
    end = cleaned.rfind('}') + 1
    if start != -1 and end > 0:
        cleaned = cleaned[start:end]

    # Убираем лишние пробелы
    cleaned = ' '.join(cleaned.split())

    return cleaned

In [ ]:
def llmcall(model: str, prompt: str, system: str, temperature: float = 0.2) -> str:
    llm = None

    if model == "giga":
        from langchain_gigachat import GigaChat
        llm = GigaChat(
            credentials="",
            verify_ssl_certs=False,
            model="GigaChat-max"
        )
    elif "llama" in model or "qwen" in model:
        from langchain_groq import ChatGroq
        model_map = {
            "llama-70b": "llama-3.3-70b-versatile",
            "llama-17b-maverick": "meta-llama/llama-4-maverick-17b-128e-instruct",
            "llama-17b-scout": "meta-llama/llama-4-scout-17b-16e-instruct",
            "qwen-32b": "qwen/qwen3-32b"
        }
        groq_model = model_map.get(model, "llama-3.1-8b-instant")

        llm = ChatGroq(
            model=groq_model,
            api_key="",
            temperature=temperature
        )
    else:
        raise ValueError(f"Неизвестная модель: {model}")

    response = llm.invoke(f"{system}\n\n{prompt}")
    return response.content


def pcm_to_list(pcm: np.ndarray):
    """Матрица numpy -> список списков для JSON/LLM."""
    return pcm.tolist()
import re
def parse_ahp_pcm(raw: str, n: int) -> np.ndarray:
    """Извлекает ТОЛЬКО матрицу из JSON"""

    # 1. Находим блок pcm
    pcm_match = re.search(r'"pcm"\s*:\s*\[([^\]]*(\[[^\]]*\][^\]]*){' + str(n) + r',})', raw, re.DOTALL)

    if not pcm_match:
        # Fallback - ищем массив из n строк
        array_match = re.search(r'\[\s*(\[[^\]]*\],\s*){' + str(n-1) + r'}\s*\[[^\]]*\]\s*\]', raw, re.DOTALL)
        if array_match:
            pcm_text = array_match.group(0)
        else:
            pcm_text = None
    else:
        pcm_text = pcm_match.group(1)

    if not pcm_text:
        raise ValueError("PCM массив не найден")

    # 2. Извлекаем числа ТОЛЬКО из pcm
    numbers = re.findall(r'1/(?:2|3|4|5|6|7|8|9)|(?:2|3|4|5|6|7|8|9)|1', pcm_text)

    if len(numbers) < n*n:
        print(f"⚠️ Найдено {len(numbers)} чисел из {n*n}, дополняем...")
        numbers += ['1'] * (n*n - len(numbers))

    # 3. Конвертируем
    pcm_flat = []
    for num in numbers[:n*n]:
        if '/' in num:
            a, b = map(int, num.split('/'))
            pcm_flat.append(a/b)
        else:
            pcm_flat.append(float(num))

    pcm = np.array(pcm_flat).reshape(n, n)
    np.fill_diagonal(pcm, 1.0)

    ci, cr = PeerToPeerConsensusAHP().check_consistency(pcm)
    print(f"✅ Матрица {n}x{n}, CR={cr:.4f}")

    return pcm


In [ ]:
def build_happiness_pcm_prompt(alternatives: List[str], country_profiles: List[str], history: str = "") -> str:
    """Промпт для ПОЛНОЙ матрицы парных сравнений"""
    profile_text = "\n".join([f"{alt}: {profile}" for alt, profile in zip(alternatives, country_profiles)])
    n = len(alternatives)

    # Шаблон пустой матрицы
    matrix_template = ""
    for i in range(n):
        row = "X"+str(i)+"["
        for j in range(n):
            if i == j:
                row += "1"
            else:
                row += "?"
            row += ", " if j < n-1 else "]"
        matrix_template += row + ",\n  "
    matrix_template = matrix_template.rstrip(",\n  ") + "\n"

    prompt = f"""{history if history else ''}
Compare countries by POPULATION HAPPINESS LEVEL using data

Country data:
{profile_text}

Build COMPLETE PAIRWISE COMPARISON MATRIX (PCM) using Saaty scale (1-9).
**MATRIX READS BY ROWS: each row = one country vs ALL others!**
First line for X1, second - X2 etc
Explain BRIEFLY — only 2-4 sentences, no detailed reasoning.

CRITICAL REQUIREMENTS:
1. FULL MATRIX {n}x{n} - ALL elements, diagonal=1
2. Follow TRIAD RULE: if A>B, B>C then A>C
3. Use ONLY Saaty scale (1-9), no *, +, -
4. Check position: if X5 better X1 → high scale on position 51

**OUTPUT ONLY VALID JSON — no other text!**
{{
  "pcm": [
    {matrix_template}
  ],
  "explain": "short explanation (2-4 sentences max)"
}}"""

    return prompt

In [ ]:
def print_final_comparison_table(self, results: Dict[str, Any]):
    alternatives = results['alternatives']
    dm_ids = results['dm_ids']
    final_pcms = results['final_pcms']
    weights = results['weights']
    group_pcm = results['group_pcm']
    group_priorities = results['group_priorities']

    n_alternatives = len(alternatives)
    n_dms = len(dm_ids)

    print(f"ГРУППОВАЯ МАТРИЦА ПАРНЫХ СРАВНЕНИЙ:")
    self.print_formatted_matrix(group_pcm, alternatives)

    print(f"\nГРУППОВЫЕ ПРИОРИТЕТЫ:")
    for i, (alt, priority) in enumerate(zip(alternatives, group_priorities)):
        print(f"   {i+1:2d}. {alt:<15} : {priority:.4f} ({priority*100:.1f}%)")

    ranked_indices = np.argsort(-group_priorities)
    print(f"\nРАНЖИРОВАНИЕ АЛЬТЕРНАТИВ:")
    for rank, idx in enumerate(ranked_indices, 1):
        print(f"   {rank:2d} место: {alternatives[idx]:<15}")

def print_formatted_matrix(self, matrix: np.ndarray, alternatives: List[str]):
    n = len(alternatives)
    header = "         " + "".join([f"{alt:>12}" for alt in alternatives])
    print(header)
    print("         " + "-" * (12 * n))

    for i in range(n):
        row_str = f"{alternatives[i]:>8} |"
        for j in range(n):
            row_str += f"{matrix[i,j]:>12.4f}"
        print(row_str)

def print_comprehensive_results(self, results: Dict[str, Any]):
    print(f"{'='*120}")

    print(f"\nОСНОВНЫЕ РЕЗУЛЬТАТЫ:")
    print(f"   Консенсус достигнут: {'ДА' if results['consensus_achieved'] else 'НЕТ'}")
    print(f"   Количество итераций: {results['iterations_used']}")
    print(f"   Финальный максимальный ICI: {np.nanmax(results['final_ici_matrix']):.4f}")
    print(f"   Пороговое значение: {self.threshold:.4f}")

    print(f"\nВЕСА ЭКСПЕРТОВ:")
    for i, (dm_id, weight) in enumerate(zip(results['dm_ids'], results['weights'])):
        print(f"   {dm_id}: {weight:.4f} ({weight*100:.1f}%)")

    self.print_final_comparison_table(results)

def run_comprehensive_interactive_analysis(self, file_path: str) -> Dict[str, Any]:
    """
    Комплексный интерактивный анализ с полным выводом результатов
    """
    # Запускаем интерактивный режим
    results = self.run_interactive_consensus_from_file(file_path)

    # Выводим полный отчет
    self.print_comprehensive_results(results)

    return results

# Добавляем методы к классу
PeerToPeerConsensusAHP.print_final_comparison_table = print_final_comparison_table
PeerToPeerConsensusAHP.print_formatted_matrix = print_formatted_matrix
PeerToPeerConsensusAHP.print_comprehensive_results = print_comprehensive_results
PeerToPeerConsensusAHP.run_comprehensive_interactive_analysis = run_comprehensive_interactive_analysis

In [ ]:
LLMMODELS = [
    "giga",                    # GigaChat-Max (твоя)
    "llama-70b",              # llama-3.3-70b-versatile
    "llama-17b-maverick",     # llama-4-maverick-17b-128e-instruct
    "qwen-32b"                # qwen/qwen3-32b
]

# Температуры для каждой модели
MODEL_TEMPS = {
    "giga": 0.3,
    "llama-70b": 0.4,          # Низкая для точности
    "llama-17b-maverick": 0.15,
    "qwen-32b": 0.2
}

In [ ]:
def llm_accept_change(model_name: str, dm_id: str,
                     old_pcm: np.ndarray, new_pcm: np.ndarray,
                     alternatives: list[str], iteration: int = 0,
                     alpha: float = None) -> Tuple[bool, str]:

    system = system = """You are a demography expert building AHP Pairwise Comparison Matrices (PCM).
ALWAYS return STRICTLY VALID JSON."""

    # альфа из алгоритма (0-1, сколько сохраняешь своего мнения)
    alpha_info = f"alpha={alpha:.3f}" if alpha is not None else ""

    prompt = build_decision_prompt(alternatives, old_pcm, new_pcm, alpha_info)

    raw = llmcall(model=model_name, prompt=prompt, system=system,
                   temperature=MODEL_TEMPS.get(model_name, 0.3))
    print(raw)
    cleaned = clean_llm_response(raw)

    try:
        data = json.loads(cleaned)
        agree = bool(data.get("agree", False))
        reason = str(data.get("reason", "No reason"))
        print(f"Parsed: agree={agree}, reason={reason[:100]}...")
        return agree, reason

    except json.JSONDecodeError as e:
        print(f"JSON Error: {e}")
        print(f"Cleaned content: {repr(cleaned[:200])}...")
        return True, f"JSON parsing failed"

In [ ]:
def build_decision_prompt(alternatives: list[str], old_pcm: np.ndarray,
                         new_pcm: np.ndarray, alpha_info: str = "") -> str:
    """
    Creates prompt for accepting/rejecting PCM changes based on country data.
    """
    n = len(alternatives)

    # 1. Extract country names from alternatives (X1 -> Cambodia, etc.)
    country_names = []
    for alt in alternatives:
        if " " in alt:
            country = alt.split(" ", 1)[1]
        else:
            country = alt.replace("X", "").strip()
        country_names.append(country)

    # 2. Country data from global COUNTRYPROFILES
    country_data = []
    for i, profile in enumerate(COUNTRY_PROFILES):
        country_data.append(f"**{country_names[i]}**: {profile}")


    # 4. Format matrices
    def pcm_to_str(pcm: np.ndarray, alternatives: list[str]) -> str:
        lines = []
        for i in range(n):
            row = [f"{alternatives[i]}"]
            for j in range(n):
                val = pcm[i,j]
                if abs(val - 1) < 0.01:
                    row.append("1")
                else:
                    row.append(f"{val:.1f}")
            lines.append(" | ".join(row))
        return "\n".join(lines)

    # 5. Main prompt
    prompt = f"""
You are a demography expert evaluating **Pairwise Comparison Matrices (PCM)** for Analytic Hierarchy Process (AHP).

=== COUNTRY DATA (unchanged reference) ===
{chr(10).join(country_data)}

=== YOUR ORIGINAL MATRIX (old_pcm) ===
{pcm_to_str(old_pcm, alternatives)}

=== PROPOSED MATRIX (new_pcm) ===
{pcm_to_str(new_pcm, alternatives)}


=== ALGORITHM ALPHA ===
{alpha_info} (how much of your original opinion to keep)

**TASK**: Logically evaluate changes based on COUNTRY DATA.
- AGREE if changes are LOGICALLY CONSISTENT with data or if alpha > 0.75 you also should agree with changes
- REJECT if changes contradict your original assessments


RESPOND ONLY with VALID JSON:
{{"agree": true/false, "reason": "short explanation (1-2 sentences)"}}
"""

    return prompt.strip()

In [ ]:

def add_to_history(model: str, role: str, content: str):
    """Добавляет сообщение в историю модели"""
    DIALOG_HISTORIES[model].append({"role": role, "content": content})
    # Ограничиваем историю (последние 12 сообщений)
    if len(DIALOG_HISTORIES[model]) > 12:
        DIALOG_HISTORIES[model] = DIALOG_HISTORIES[model][-12:]

def get_history_context(model: str, max_msgs: int = 8) -> str:
    """Возвращает контекст истории для промпта"""
    history = DIALOG_HISTORIES[model][-max_msgs:]
    if not history:
        return ""

    context = "=== PREVIOUS INTERACTIONS ===\n"
    for msg in reversed(history):
        context += f"{msg['role'].upper()}: {msg['content'][:200]}...\n"
    return context + "\n"

In [ ]:
def run_interactive_consensus_from_llms(self, alternatives: List[str], models: List[str] = None) -> Dict[str, Any]:
    """
    Тот же алгоритм консенсуса, что и run_interactive_consensus_from_file,
    но исходные матрицы задаются LLM-экспертами.
    """
    # 1. Инициализация от LLM
    #self.init_pcms_from_llms(alternatives, models)

    print(f"Сгенерировано {len(self.current_pcms)} матриц PCM от LLM-экспертов: {self.dm_ids}")
    print(f"Альтернативы: {self.alternatives}")
    print(f"Порог консенсуса: {self.threshold}")
    print(f"Максимальное число итераций: {self.max_iterations}")

    print(f"\nНАЧАЛЬНЫЕ МАТРИЦЫ ЭКСПЕРТОВ:")
    for i, pcm in enumerate(self.current_pcms):
        ci, cr = self.check_consistency(pcm)
        status = "Приемлема" if cr < 0.1 else "Неприемлема"
        print(f"\n{self.dm_ids[i]} (CI={ci:.4f}, CR={cr:.4f} - {status}):")
        self.print_simple_matrix(pcm)

    m = len(self.current_pcms)
    D: Set[Tuple[int, int]] = set((i, j) for i in range(m) for j in range(m) if i != j)
    history = []

    # 2. Главный цикл консенсуса (как в run_interactive_consensus_from_file)
    for t in range(self.max_iterations):
        ici_matrix = self.calculate_ici_matrix(self.current_pcms, D)
        max_ici = np.nanmax(ici_matrix)

        # Условия остановки
        if max_ici <= self.threshold:
            print(f"\nДОСТИГНУТ КОНСЕНСУС! Максимальный ICI = {max_ici:.4f} ≤ {self.threshold:.4f}")
            break
        elif len(D) == 0:
            print(f"\nВСЕ ЭКСПЕРТЫ ОТКАЗАЛИСЬ ОТ ИЗМЕНЕНИЙ")
            break

        selected_pair = self.find_max_ici_pair(ici_matrix, D)
        if selected_pair is None:
            break

        a, b = selected_pair

        # Опрос LLM-экспертов (новая версия функции)
        accept_a, accept_b = self.enhanced_interactive_acceptance_decision(
            selected_pair, t, self.dm_ids, ici_matrix, D
        )

        alpha_a, alpha_b = self.calculate_alpha(ici_matrix, selected_pair, m)

        if accept_a and accept_b:
            A_a_temp = self.current_pcms[a].copy()
            A_b_temp = self.current_pcms[b].copy()

            self.current_pcms[a] = self.update_pcm(A_a_temp, A_b_temp, alpha_a)
            self.current_pcms[b] = self.update_pcm(A_b_temp, A_a_temp, alpha_b)

        elif accept_a and not accept_b:
            A_a_temp = self.current_pcms[a].copy()
            A_b_temp = self.current_pcms[b].copy()
            self.current_pcms[a] = self.update_pcm(A_a_temp, A_b_temp, alpha_a)

            D.discard((b, a))
            print(f" Удалена направленная пара: ({self.dm_ids[b]}, {self.dm_ids[a]})")

        elif not accept_a and accept_b:
            A_a_temp = self.current_pcms[a].copy()
            A_b_temp = self.current_pcms[b].copy()

            self.current_pcms[b] = self.update_pcm(A_b_temp, A_a_temp, alpha_b)

            D.discard((a, b))
            print(f"Удалена направленная пара: ({self.dm_ids[a]}, {self.dm_ids[b]})")

        else:
            D.discard((a, b))
            D.discard((b, a))
            print(f"Удалены направленные пары: ({self.dm_ids[a]}, {self.dm_ids[b]}) и ({self.dm_ids[b]}, {self.dm_ids[a]})")

    # 3. Финальные расчёты (как в оригинале)
    final_ici_matrix = self.calculate_ici_matrix(self.current_pcms, D)
    weights = self.calculate_decision_maker_weights(final_ici_matrix, m)
    group_pcm = self.aggregate_group_pcm(self.current_pcms, weights)
    group_priorities = self.calculate_priority_vector(group_pcm)

    print(f"\n{'='*80}")
    print(f"ФИНАЛЬНАЯ МАТРИЦА ICI:")
    self.print_ici_matrix(final_ici_matrix, self.dm_ids)

    results = {
        "final_pcms": self.current_pcms,
        "final_ici_matrix": final_ici_matrix,
        "group_pcm": group_pcm,
        "group_priorities": group_priorities,
        "weights": weights,
        "iterations_used": t + 1,
        "final_D": D,
        "history": history,
        "consensus_achieved": np.nanmax(final_ici_matrix) <= self.threshold,
        "alternatives": self.alternatives,
        "dm_ids": self.dm_ids,
        "parameters": {},  # при желании добавьте свои параметры
    }

    return results

PeerToPeerConsensusAHP.run_interactive_consensus_from_llms = run_interactive_consensus_from_llms

# Запуск

In [ ]:
def load_happiness_countries(csv_path: str = "World-happiness-report-2024.csv",
                           n_countries: int = 5,
                           filter_type: str = "top",  # "top" | "bottom"
                           shuffle_data: bool = True) -> tuple:
    """ТОП или ДНО + ПЕРЕМЕШИВАНИЕ ДАННЫХ"""

    df = pd.read_csv(csv_path)

    KEY_COLS = {
        'Country name': 'Country name', 'Ladder score': 'Ladder score',
        'GDP': 'Log GDP per capita', 'Social_support': 'Social support',
        'Healthy_life_exp': 'Healthy life expectancy',
        'Freedom': 'Freedom to make life choices', 'Generosity': 'Generosity',
        'Corruption': 'Perceptions of corruption'
    }

    df_clean = df.dropna(subset=['Ladder score'])

    # ФИЛЬТРАЦИЯ
    if filter_type == "top":
        selected_data = df_clean.nlargest(n_countries, 'Ladder score')
        print(f"📈 ТОП-{n_countries} счастливых")
    elif filter_type == "bottom":
        selected_data = df_clean.nsmallest(n_countries, 'Ladder score')
        print(f"📉 {n_countries} самых несчастных")
    else:
        raise ValueError("filter_type: 'top' или 'bottom'")

    selected_data = selected_data.reset_index(drop=True)

    # ПЕРЕМЕШИВАНИЕ ДАННЫХ (не только вывода!)
    if shuffle_data:
        indices = list(range(len(selected_data)))
        random.seed(2)
        random.shuffle(indices)
        selected_data = selected_data.iloc[indices].reset_index(drop=True)
        print("🔀 Данные перемешаны")

    alternatives = []
    country_profiles = []
    true_happiness_scores = []

    for i, row in selected_data.iterrows():
        country = row['Country name']
        true_happiness_scores.append(row['Ladder score'])
        alternatives.append(f"X{i+1}: {country}")

        profile = f"{country}: GDP/capita={row['Log GDP per capita']:.2f}, Social_support={row['Social support']:.3f}, Healthy_life={row['Healthy life expectancy']:.1f}yr, Freedom={row['Freedom to make life choices']:.3f}, Generosity={row['Generosity']:+.3f}, Corruption={row['Perceptions of corruption']:.3f}"
        country_profiles.append(profile)

    # ВЫВОД
    print(f"\n✅ {len(alternatives)} стран ({filter_type}, shuffle={shuffle_data}):")
    for alt, profile in zip(alternatives, country_profiles):
        print(f"  {alt}")
        print(f"    {profile}\n")

    print("💡 True scores (скрыты):")
    for country, score in zip([a.split(': ')[1] for a in alternatives], true_happiness_scores):
        print(f"  {country}: {score:.3f}")

    random.seed(-1)
    return alternatives, country_profiles, true_happiness_scores

In [ ]:
ALTERNATIVES, COUNTRY_PROFILES, TRUE_SCORES = load_happiness_countries(n_countries=5, filter_type="top", shuffle_data=True)

📈 ТОП-5 счастливых
🔀 Данные перемешаны

✅ 5 стран (top, shuffle=True):
  X1: Iceland
    Iceland: GDP/capita=1.88, Social_support=1.617, Healthy_life=0.7yr, Freedom=0.819, Generosity=+0.258, Corruption=0.182

  X2: Denmark
    Denmark: GDP/capita=1.91, Social_support=1.520, Healthy_life=0.7yr, Freedom=0.823, Generosity=+0.204, Corruption=0.548

  X3: Sweden
    Sweden: GDP/capita=1.88, Social_support=1.501, Healthy_life=0.7yr, Freedom=0.838, Generosity=+0.221, Corruption=0.524

  X4: Israel
    Israel: GDP/capita=1.80, Social_support=1.513, Healthy_life=0.7yr, Freedom=0.641, Generosity=+0.153, Corruption=0.193

  X5: Finland
    Finland: GDP/capita=1.84, Social_support=1.572, Healthy_life=0.7yr, Freedom=0.859, Generosity=+0.142, Corruption=0.546

💡 True scores (скрыты):
  Iceland: 7.525
  Denmark: 7.583
  Sweden: 7.344
  Israel: 7.341
  Finland: 7.741


In [ ]:

def gen_model(consensus, modelname: str, alternatives: list, country_profiles: list, max_retries=10):
    """Генерирует PCM для одной модели и сохраняет в глобальный кэш"""
    global PCM_CACHE, CRS_CACHE

    print(f"\n🚀 {modelname}...")
    n = len(alternatives)

    system = """You are a demography expert building AHP Pairwise Comparison Matrices (PCM).
ALWAYS return STRICTLY VALID JSON with FULL reciprocal matrix."""
    temp = MODEL_TEMPS.get(modelname, 0.3)

    best_pcm = None
    best_cr = float('inf')
    attempt = 0

    while attempt < max_retries:
        attempt += 1
        print(f"  Attempt {attempt}/{max_retries}")

        history = get_history_context(modelname)
        feedback = ""
        if attempt > 1 and best_pcm is not None:
            feedback = f"""
PREVIOUS MATRIX CR={best_cr:.3f} > 0.1

IMPROVE CONSISTENCY:
- Change 1-2 comparisons by 1-2 points only
- Focus on TRIAD consistency (a_ik ≈ a_ij * a_jk)
- Keep overall ranking intact"""

        prompt = build_happiness_pcm_prompt(alternatives, country_profiles, history + feedback)
        add_to_history(modelname, "user", prompt)

        raw = llmcall(model=modelname, prompt=prompt, system=system, temperature=temp)
        print(f"  Raw: {raw}...")
        add_to_history(modelname, "assistant", raw)

        try:
            pcm_candidate = parse_ahp_pcm(raw, n)
            if pcm_candidate.shape != (n, n):
                raise ValueError(f"Shape {pcm_candidate.shape} != ({n},{n})")

            ci, cr = consensus.check_consistency(pcm_candidate)
            print(f"  CR={cr:.4f}")

            if cr < best_cr:
                best_cr = cr
                best_pcm = pcm_candidate.copy()

            if cr < 0.1:
                print(" CR OK!")
                pcm = pcm_candidate
                break

        except Exception as e:
            print(f"Parse error: {e}")

    # Fallback
    if 'pcm' not in locals():
        if best_pcm is not None:
            print(f"  Fixing best CR={best_cr:.4f}...")
            pcm = consensus.fix_pcm_by_eigenvector(best_pcm, max_changes=3)
            _, cr_fixed = consensus.check_consistency(pcm)
            print(f"  Fixed CR={cr_fixed:.4f}")
        else:
            print("  Fallback: identity matrix")
            pcm = np.eye(n)

    # Сохранение в глобальный кэш
    PCM_CACHE[modelname] = pcm.copy()
    CRS_CACHE[modelname] = consensus.check_consistency(pcm)[1]
    print(f" {modelname} ready: CR={CRS_CACHE[modelname]:.4f}")

In [ ]:
DIALOG_HISTORIES = {model: [] for model in LLMMODELS}
PCM_CACHE = {}
CRS_CACHE = {}

In [ ]:
DIALOG_HISTORIES = {model: [] for model in LLMMODELS}

In [ ]:
consensus = PeerToPeerConsensusAHP(threshold=1.03, max_iterations=20)
gen_model(consensus, "giga", ALTERNATIVES, COUNTRY_PROFILES)


🚀 giga...
  Attempt 1/10
  Raw: {
  "pcm": [
    [1, 1/1.1, 1.1, 1.2, 1/1.1], 
    [1.1, 1, 1.2, 1.3, 1], 
    [1/1.1, 1/1.2, 1, 1.1, 1/1.1], 
    [1/1.2, 1/1.3, 1/1.1, 1, 1/1.2], 
    [1.1, 1, 1.1, 1.2, 1]
  ],
  "explain": "Iceland is slightly preferred over Sweden and Denmark due to higher social support & freedom scores. Israel lags behind in happiness factors leading to lower ratings."
}...
✅ Матрица 5x5, CR=0.1580
  CR=0.1580
  Attempt 2/10


KeyboardInterrupt: 

In [ ]:
gen_model(consensus, "llama-70b", ALTERNATIVES, COUNTRY_PROFILES)


🚀 llama-70b...
  Attempt 1/10
  Raw: ```json
{
  "pcm": [
    [1, 4, 3, 5, 2],
    [1/4, 1, 1/2, 3, 1/3],
    [1/3, 2, 1, 5, 1/2],
    [1/5, 1/3, 1/5, 1, 1/4],
    [1/2, 3, 2, 4, 1]
  ],
  "explain": "The comparison is based on the overall happiness level, considering factors like GDP, social support, and freedom. Iceland is compared favorably to others due to its high social support and low corruption. The matrix follows the Saaty scale and the triad rule, ensuring consistency in the comparisons."
}
```...
✅ Матрица 5x5, CR=0.0386
  CR=0.0386
 CR OK!
 llama-70b ready: CR=0.0386


In [ ]:
gen_model(consensus, "llama-17b-maverick", ALTERNATIVES, COUNTRY_PROFILES)


🚀 llama-17b-maverick...
  Attempt 1/10
  Raw: To generate the pairwise comparison matrix (PCM) for the given countries based on their population happiness level, we first need to understand the factors that contribute to the happiness level as per the given data. The factors include GDP/capita, Social support, Healthy life, Freedom, Generosity, and Corruption. We will compare the countries based on these criteria.

Let's analyze the given data:
- X1 (Iceland): GDP/capita = 1.88, Social_support = 1.617, Healthy_life = 0.7, Freedom = 0.819, Generosity = 0.258, Corruption = 0.182
- X2 (Denmark): GDP/capita = 1.91, Social_support = 1.520, Healthy_life = 0.7, Freedom = 0.823, Generosity = 0.204, Corruption = 0.548
- X3 (Sweden): GDP/capita = 1.88, Social_support = 1.501, Healthy_life = 0.7, Freedom = 0.838, Generosity = 0.221, Corruption = 0.524
- X4 (Israel): GDP/capita = 1.80, Social_support = 1.513, Healthy_life = 0.7, Freedom = 0.641, Generosity = 0.153, Corruption = 0.193
- X5 (Finlan

In [ ]:
gen_model(consensus, "qwen-32b", ALTERNATIVES, COUNTRY_PROFILES)


🚀 qwen-32b...
  Attempt 1/10
  Raw: <think>
Okay, let's tackle this problem step by step. The user wants a pairwise comparison matrix (PCM) using the Saaty scale (1-9) based on the population happiness level of five countries. The data provided includes various indicators like GDP per capita, social support, healthy life, freedom, generosity, and corruption. 

First, I need to understand how to compare these countries based on their happiness. Since the exact happiness scores aren't given, I'll have to infer them from the provided indicators. Each country's overall happiness might be a combination of these factors. For example, higher GDP per capita and social support likely contribute to higher happiness, while lower corruption and higher freedom might also play a role.

Let me list out the countries again to keep track:

1. Lesotho (X1)
2. Lebanon (X2)
3. Sierra Leone (X3)
4. Congo (Kinshasa) (X4)
5. Afghanistan (X5)

Now, I need to compare each country against every other using the

In [ ]:
consensus.alternatives = ALTERNATIVES
consensus.dm_ids = list(PCM_CACHE.keys())
consensus.current_pcms = list(PCM_CACHE.values())

In [ ]:
import json
from datetime import datetime

def save_init_pcms_to_json(model, filename: str = "init_happiness_pcms.json"):
    """
    Сохраняет ТОЛЬКО матрицы из self.current_pcms после init_pcms_from_llms()

    Args:
        model: PeerToPeerConsensusAHP после init_pcms_from_llms()
        filename: имя файла
    """
    output = {
        "alternatives": model.alternatives,
        "dms": [
            {
                "id": dm_id,
                "pcm": pcm.tolist(),  # numpy → list с дробями 1/3, 0.333
                "cr": model.check_consistency(pcm)[1]  # CR для каждой матрицы
            }
            for dm_id, pcm in zip(model.dm_ids, model.current_pcms)
        ],
        "parameters": {
            "n_countries": len(model.alternatives),
            "n_models": len(model.dm_ids),
            "true_happiness_scores": TRUE_SCORES  # ground truth
        },
        "generated": datetime.now().isoformat(),
        "method": "init_pcms_from_llms"
    }

    # ✅ ТОЧНО как example_pcm.json
    with open(filename, 'w', encoding='utf-8') as f:
        json.dump(output, f, indent=2, ensure_ascii=False)

    print(f"✅ Сохранено {len(output['dms'])} PCM в {filename}")

In [ ]:
save_init_pcms_to_json(consensus)

✅ Сохранено 4 PCM в init_happiness_pcms.json


In [ ]:
results = consensus.run_interactive_consensus_from_llms(ALTERNATIVES, LLMMODELS)



Сгенерировано 4 матриц PCM от LLM-экспертов: ['giga', 'llama-70b', 'llama-17b-maverick', 'qwen-32b']
Альтернативы: ['X1: Lesotho', 'X2: Lebanon', 'X3: Sierra Leone', 'X4: Congo (Kinshasa)', 'X5: Afghanistan']
Порог консенсуса: 1.03
Максимальное число итераций: 20

НАЧАЛЬНЫЕ МАТРИЦЫ ЭКСПЕРТОВ:

giga (CI=0.0321, CR=0.0289 - Приемлема):
      1.000   0.333   2.000   3.000   5.000
      3.000   1.000   5.000   6.000   8.000
      0.500   0.200   1.000   2.000   4.000
      0.333   0.167   0.500   1.000   3.000
      0.200   0.125   0.250   0.333   1.000

llama-70b (CI=0.0170, CR=0.0153 - Приемлема):
      1.000   2.000   4.000   3.000   5.000
      0.500   1.000   3.000   2.000   4.000
      0.250   0.333   1.000   0.500   2.000
      0.333   0.500   2.000   1.000   3.000
      0.200   0.250   0.500   0.333   1.000

llama-17b-maverick (CI=0.0228, CR=0.0205 - Приемлема):
      1.000   0.333   2.000   2.000   3.000
      3.000   1.000   5.000   5.000   7.000
      0.500   0.200   1.000   1.0

In [ ]:
consensus.print_comprehensive_results(results)


ОСНОВНЫЕ РЕЗУЛЬТАТЫ:
   Консенсус достигнут: ДА
   Количество итераций: 8
   Финальный максимальный ICI: 1.0253
   Пороговое значение: 1.0300

ВЕСА ЭКСПЕРТОВ:
   giga: 0.1881 (18.8%)
   llama-70b: 0.2489 (24.9%)
   llama-17b-maverick: 0.2814 (28.1%)
   qwen-32b: 0.2815 (28.2%)
ГРУППОВАЯ МАТРИЦА ПАРНЫХ СРАВНЕНИЙ:
          X1: Lesotho X2: LebanonX3: Sierra LeoneX4: Congo (Kinshasa)X5: Afghanistan
         ------------------------------------------------------------
X1: Lesotho |      1.0000      0.9720      2.5015      2.4393      4.6566
X2: Lebanon |      1.0291      1.0000      2.5341      2.2925      4.9437
X3: Sierra Leone |      0.3997      0.3945      1.0000      0.8972      3.0954
X4: Congo (Kinshasa) |      0.4094      0.4361      1.1143      1.0000      3.1991
X5: Afghanistan |      0.2148      0.2025      0.3230      0.3128      1.0000

ГРУППОВЫЕ ПРИОРИТЕТЫ:
    1. X1: Lesotho     : 0.3230 (32.3%)
    2. X2: Lebanon     : 0.3280 (32.8%)
    3. X3: Sierra Leone : 0.1410 (14.1%

# **КОНТРАСТ**

In [ ]:
def load_mixed_gdp_countries(csv_path: str = "World-happiness-report-2024.csv",
                            n_countries: int = 3,
                            shuffle_data: bool = True) -> tuple:
    """ТОП-3 + ДНО-3 по GDP = 6 стран ВМЕСТЕ + ПЕРЕМЕШИВАНИЕ"""

    df = pd.read_csv(csv_path)
    df_clean = df.dropna(subset=['Ladder score', 'Log GDP per capita'])

    # ТОП-3 и ДНО-3 по GDP
    top_gdp = df_clean.nlargest(n_countries, 'Log GDP per capita')
    bottom_gdp = df_clean.nsmallest(3, 'Log GDP per capita')

    # ОБЪЕДИНЯЕМ в одну выборку 6 стран
    selected_data = pd.concat([top_gdp, bottom_gdp]).drop_duplicates().reset_index(drop=True)

    print(f"💎 ТОП-{n_countries} + ДНО-{n_countries} по GDP = {len(selected_data)} стран")

    # ПЕРЕМЕШИВАНИЕ
    if shuffle_data:
        indices = list(range(len(selected_data)))
        random.seed(42)
        random.shuffle(indices)
        selected_data = selected_data.iloc[indices].reset_index(drop=True)
        print("🔀 Данные перемешаны")

    alternatives = []
    country_profiles = []
    true_happiness_scores = []

    for i, row in selected_data.iterrows():
        country = row['Country name']
        true_happiness_scores.append(row['Ladder score'])
        alternatives.append(f"X{i+1}: {country}")

        profile = f"{country}: GDP/capita={row['Log GDP per capita']:.2f}, Social_support={row['Social support']:.3f}, Healthy_life={row['Healthy life expectancy']:.1f}yr, Freedom={row['Freedom to make life choices']:.3f}, Generosity={row['Generosity']:+.3f}, Corruption={row['Perceptions of corruption']:.3f}"
        country_profiles.append(profile)

    # ВЫВОД
    print(f"\n✅ {len(alternatives)} стран (mixed GDP extremes, shuffle={shuffle_data}):")
    for alt, profile in zip(alternatives, country_profiles):
        print(f"  {alt}")
        print(f"    {profile}\n")

    print("💡 True scores (скрыты):")
    for country, score in zip([a.split(': ')[1] for a in alternatives], true_happiness_scores):
        print(f"  {country}: {score:.3f}")

    random.seed(-1)
    return alternatives, country_profiles, true_happiness_scores

In [ ]:
def load_mixed_soc_countries(csv_path: str = "World-happiness-report-2024.csv",
                            n_countries: int = 3,
                            shuffle_data: bool = True) -> tuple:
    """ТОП-3 + ДНО-3 по Social support """

    df = pd.read_csv(csv_path)
    df_clean = df.dropna(subset=['Ladder score', 'Social support'])

    # ТОП-3 и ДНО-3 по GDP
    top_gdp = df_clean.nlargest(2, 'Social support')
    bottom_gdp = df_clean.nsmallest(3, 'Social support')

    # ОБЪЕДИНЯЕМ в одну выборку 6 стран
    selected_data = pd.concat([top_gdp, bottom_gdp]).drop_duplicates().reset_index(drop=True)

    print(f"💎 ТОП-{n_countries} + ДНО-{n_countries} по Social support = {len(selected_data)} стран")

    # ПЕРЕМЕШИВАНИЕ
    if shuffle_data:
        indices = list(range(len(selected_data)))
        random.seed(42)
        random.shuffle(indices)
        selected_data = selected_data.iloc[indices].reset_index(drop=True)
        print("🔀 Данные перемешаны")

    alternatives = []
    country_profiles = []
    true_happiness_scores = []

    for i, row in selected_data.iterrows():
        country = row['Country name']
        true_happiness_scores.append(row['Ladder score'])
        alternatives.append(f"X{i+1}: {country}")

        profile = f"{country}: GDP/capita={row['Log GDP per capita']:.2f}, Social_support={row['Social support']:.3f}, Healthy_life={row['Healthy life expectancy']:.1f}yr, Freedom={row['Freedom to make life choices']:.3f}, Generosity={row['Generosity']:+.3f}, Corruption={row['Perceptions of corruption']:.3f}"
        country_profiles.append(profile)

    # ВЫВОД
    print(f"\n✅ {len(alternatives)} стран :")
    for alt, profile in zip(alternatives, country_profiles):
        print(f"  {alt}")
        print(f"    {profile}\n")

    print("💡 True scores (скрыты):")
    for country, score in zip([a.split(': ')[1] for a in alternatives], true_happiness_scores):
        print(f"  {country}: {score:.3f}")

    random.seed(-1)
    return alternatives, country_profiles, true_happiness_scores

In [ ]:
def load_specific_countries(csv_path: str = "World-happiness-report-2024.csv") -> tuple:
    """Фиксированные 5 стран: """
    df = pd.read_csv(csv_path)

    # ТОЧНЫЙ ПОРЯДОК ДЛЯ X1-X5
    country_order =  ['Afghanistan','Bangladesh', 'Luxembourg', 'Benin', 'Venezuela']

    selected_rows = []
    for country in country_order:
        row = df[df['Country name'] == country].iloc[0]
        selected_rows.append(row)

    selected_data = pd.DataFrame(selected_rows).reset_index(drop=True)

    alternatives = [f"X{i+1}: {row['Country name']}" for i, row in selected_data.iterrows()]
    country_profiles = []
    true_happiness_scores = selected_data['Ladder score'].tolist()

    print("✅ Данные из CSV (X1-X5):")
    for i, row in selected_data.iterrows():
        profile = (f"{row['Country name']}: GDP/capita={row['Log GDP per capita']:.2f}, "
                  f"Social_support={row['Social support']:.3f}, "
                  f"Healthy_life={row['Healthy life expectancy']:.1f}yr, "
                  f"Freedom={row['Freedom to make life choices']:.3f}, "
                  f"Generosity={row['Generosity']:+.3f}, "
                  f"Corruption={row['Perceptions of corruption']:.3f}")
        country_profiles.append(profile)
        print(f"  {alternatives[i]}")
        print(f"    {profile}\n")

    print("💡 True scores (скрыты):")
    for country, score in zip(country_order, true_happiness_scores):
        print(f"  {country}: {score:.3f}")

    return alternatives, country_profiles, true_happiness_scores

In [ ]:
ALTERNATIVES, COUNTRY_PROFILES, TRUE_SCORES = load_specific_countries()

✅ Данные из CSV (X1-X5):
  X1: Afghanistan
    Afghanistan: GDP/capita=0.63, Social_support=0.000, Healthy_life=0.2yr, Freedom=0.000, Generosity=+0.091, Corruption=0.088

  X2: Bangladesh
    Bangladesh: GDP/capita=1.12, Social_support=0.249, Healthy_life=0.5yr, Freedom=0.775, Generosity=+0.140, Corruption=0.167

  X3: Luxembourg
    Luxembourg: GDP/capita=2.14, Social_support=1.355, Healthy_life=0.7yr, Freedom=0.801, Generosity=+0.146, Corruption=0.432

  X4: Benin
    Benin: GDP/capita=0.91, Social_support=0.128, Healthy_life=0.3yr, Freedom=0.567, Generosity=+0.112, Corruption=0.252

  X5: Venezuela
    Venezuela: GDP/capita=0.00, Social_support=1.321, Healthy_life=0.5yr, Freedom=0.518, Generosity=+0.192, Corruption=0.086

💡 True scores (скрыты):
  Afghanistan: 1.721
  Bangladesh: 3.886
  Luxembourg: 7.122
  Benin: 4.377
  Venezuela: 5.607


In [ ]:
DIALOG_HISTORIES = {model: [] for model in LLMMODELS}
PCM_CACHE = {}
CRS_CACHE = {}

In [ ]:
DIALOG_HISTORIES = {model: [] for model in LLMMODELS}

In [ ]:
consensus = PeerToPeerConsensusAHP(threshold=1.03, max_iterations=20)
gen_model(consensus, "giga", ALTERNATIVES, COUNTRY_PROFILES)


🚀 giga...
  Attempt 1/10
  Raw: {
  "pcm": [
    [1,   1/3, 1/9, 1/4, 1/6], 
    [3,   1,   1/7, 1/2, 1/4],  
    [9,   7,   1,   5,   6],    
    [4,   2,   1/5, 1,   1/2],  
    [6,   4,   1/6, 2,   1]
  ],
  "explain": "Luxembourg dominates in happiness due to high social support and healthy life expectancy. Other poorer nations like Afghanistan and Benin score lower across all indicators."
}...
✅ Матрица 5x5, CR=0.0602
  CR=0.0602
 CR OK!
 giga ready: CR=0.0602


In [ ]:
gen_model(consensus, "llama-70b", ALTERNATIVES, COUNTRY_PROFILES)


🚀 llama-70b...
  Attempt 1/10
  Raw: ```json
{
  "pcm": [
    [1, 2, 1, 1, 3],
    [1/2, 1, 1/2, 1/3, 2],
    [3, 5, 1, 1/2, 6],
    [3, 6, 7, 1, 8],
    [1/3, 1/2, 1/6, 1/8, 1]
  ],
  "explain": "The comparison is based on the given country data, with Luxembourg having the highest overall rating and Afghanistan having a lower rating. Venezuela's high social support and generosity contribute to its higher rating. The matrix follows the triad rule and uses the Saaty scale."
}
```...
✅ Матрица 5x5, CR=0.3971
  CR=0.3971
  Attempt 2/10
  Raw: ```json
{
  "pcm": [
    [1, 1/5, 1/7, 1/8, 1/3],
    [5, 1, 1/3, 1/4, 2],
    [7, 3, 1, 1/2, 5],
    [8, 4, 2, 1, 6],
    [3, 1/2, 1/5, 1/6, 1]
  ],
  "explain": "The comparison is based on the given country data, focusing on population happiness level. The matrix is constructed using the Saaty scale, following the triad rule for consistency. The ranking is maintained, with Luxembourg and Benin having higher happiness levels compared to other count

In [ ]:
gen_model(consensus, "llama-17b-maverick", ALTERNATIVES, COUNTRY_PROFILES)


🚀 llama-17b-maverick...
  Attempt 1/10
  Raw: ```json
{
  "pcm": [
    [1, 1/3, 1/9, 1/2, 1/5],
    [3, 1, 1/7, 2, 1/3],
    [9, 7, 1, 8, 5],
    [2, 1/2, 1/8, 1, 1/4],
    [5, 3, 1/5, 4, 1]
  ],
  "explain": "Luxembourg is the happiest, Venezuela is happier than Afghanistan and Benin but less happy than Luxembourg, Bangladesh is moderately happy."
}
```...
✅ Матрица 5x5, CR=0.0426
  CR=0.0426
 CR OK!
 llama-17b-maverick ready: CR=0.0426


In [ ]:
gen_model(consensus, "qwen-32b", ALTERNATIVES, COUNTRY_PROFILES)


🚀 qwen-32b...
  Attempt 1/10
  Raw: <think>
Okay, let's tackle this problem. The user wants a pairwise comparison matrix (PCM) using the Saaty scale for five countries based on their population happiness levels. The data provided includes various indicators like GDP per capita, social support, healthy life, freedom, generosity, and corruption.

First, I need to compare each country against every other country in terms of their overall happiness. Since the Saaty scale uses 1-9, where 1 means equal, and higher numbers indicate stronger preference, I have to determine how each country stacks up against another.

Looking at the data, Luxembourg (X3) has the highest GDP per capita and social support, which are key factors in happiness. Venezuela (X5) has a very low GDP but high social support. Afghanistan (X1) is at the bottom in most categories except generosity. Bangladesh (X2) and Benin (X4) are in the middle.

For each pair, I'll assess which country is happier and assign a value based

In [ ]:
consensus.alternatives = ALTERNATIVES
consensus.dm_ids = list(PCM_CACHE.keys())
consensus.current_pcms = list(PCM_CACHE.values())
save_init_pcms_to_json(consensus)

✅ Сохранено 4 PCM в init_happiness_pcms.json


In [ ]:
results = consensus.run_interactive_consensus_from_llms(ALTERNATIVES, LLMMODELS)

Сгенерировано 4 матриц PCM от LLM-экспертов: ['giga', 'llama-70b', 'llama-17b-maverick', 'qwen-32b']
Альтернативы: ['X1: Afghanistan', 'X2: Bangladesh', 'X3: Luxembourg', 'X4: Benin', 'X5: Venezuela']
Порог консенсуса: 1.03
Максимальное число итераций: 20

НАЧАЛЬНЫЕ МАТРИЦЫ ЭКСПЕРТОВ:

giga (CI=0.0668, CR=0.0602 - Приемлема):
      1.000   0.333   0.111   0.250   0.167
      3.000   1.000   0.143   0.500   0.250
      9.000   7.000   1.000   5.000   6.000
      4.000   2.000   0.200   1.000   0.500
      6.000   4.000   0.167   2.000   1.000

llama-70b (CI=0.0359, CR=0.0324 - Приемлема):
      1.000   0.200   0.143   0.125   0.333
      5.000   1.000   0.333   0.250   2.000
      7.000   3.000   1.000   0.500   5.000
      8.000   4.000   2.000   1.000   6.000
      3.000   0.500   0.200   0.167   1.000

llama-17b-maverick (CI=0.0473, CR=0.0426 - Приемлема):
      1.000   0.333   0.111   0.500   0.200
      3.000   1.000   0.143   2.000   0.333
      9.000   7.000   1.000   8.000   5.0

In [ ]:
consensus.print_comprehensive_results(results)


ОСНОВНЫЕ РЕЗУЛЬТАТЫ:
   Консенсус достигнут: ДА
   Количество итераций: 12
   Финальный максимальный ICI: 1.0079
   Пороговое значение: 1.0300

ВЕСА ЭКСПЕРТОВ:
   giga: 0.4286 (42.9%)
   llama-70b: 0.1429 (14.3%)
   llama-17b-maverick: 0.2857 (28.6%)
   qwen-32b: 0.1429 (14.3%)
ГРУППОВАЯ МАТРИЦА ПАРНЫХ СРАВНЕНИЙ:
         X1: AfghanistanX2: BangladeshX3: Luxembourg   X4: BeninX5: Venezuela
         ------------------------------------------------------------
X1: Afghanistan |      1.0000      0.2918      0.1186      0.2247      0.1978
X2: Bangladesh |      3.4181      1.0000      0.1928      0.4868      0.4238
X3: Luxembourg |      8.4405      5.1987      1.0000      2.6697      2.7631
X4: Benin |      4.4534      2.0564      0.3746      1.0000      0.8338
X5: Venezuela |      5.0499      2.3580      0.3617      1.1993      1.0000

ГРУППОВЫЕ ПРИОРИТЕТЫ:
    1. X1: Afghanistan : 0.0410 (4.1%)
    2. X2: Bangladesh  : 0.0990 (9.9%)
    3. X3: Luxembourg  : 0.4710 (47.1%)
    4. X4: Beni